## ▶ Run Online — No Installation Needed

| Platform | Link |
|---|---|
| **Binder** (no account) | [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Piyushjhu/HELIX_Toolbox/main?labpath=examples%2F03_spade_spall_hel_analysis.ipynb) |
| **Google Colab** | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Piyushjhu/HELIX_Toolbox/blob/main/examples/03_spade_spall_hel_analysis.ipynb) |
| **GitHub Codespaces** | [![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/Piyushjhu/HELIX_Toolbox) |

This notebook starts from a `*--vel-smooth-with-uncert.csv` file (produced by ALPSS / Example 02). The setup cell below will first run ALPSS on the bundled sample data to generate that file automatically when no existing velocity file is provided.

# Example 3 — SPADE: Spall Strength, Strain Rate & HEL Detection

This notebook demonstrates the **SPADE** analysis stage in isolation,
starting from a pre-existing smoothed velocity + uncertainty CSV
(the `*--vel-smooth-with-uncert.csv` file produced by ALPSS).

Topics covered:
1. Load a velocity trace and visualise the raw profile
2. Run SPADE with `spade_only` mode
3. Inspect the spall detection result (5-segment fit)
4. Inspect the HEL detection result (RDP + linear regression)
5. Explore how key parameters change the result

---
**Before running:** update the path variables in the next cell.

In [ ]:
import os, sys, subprocess

# ── Cloud / Online environment setup ──────────────────────────────────────
try:
    import google.colab
    _ENV = "colab"
except ImportError:
    _ENV = "binder" if os.environ.get("BINDER_SERVICE_HOST") else "local"

if _ENV in ("colab", "binder"):
    os.environ["QT_QPA_PLATFORM"] = "offscreen"
    os.environ["MPLBACKEND"] = "Agg"

if _ENV == "colab":
    REPO_ROOT = "/content/HELIX_Toolbox"
    if not os.path.isdir(REPO_ROOT):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Piyushjhu/HELIX_Toolbox.git",
                        REPO_ROOT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    os.path.join(REPO_ROOT, "requirements.txt")], check=False)
else:
    REPO_ROOT = os.path.abspath("..")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

SAMPLE_DIR = os.path.join(REPO_ROOT, "input_data", "C1_files")
print(f"Environment: {_ENV} | REPO_ROOT: {REPO_ROOT}")
if _ENV == "colab":
    print("To upload YOUR OWN velocity CSV:  from google.colab import files; uploaded = files.upload()")
# ──────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# ── USER PATHS — edit these ────────────────────────────────────────────────
# Point VEL_FILE to a *--vel-smooth-with-uncert.csv produced by ALPSS.
# If you don't have one yet, run Example 02 first (it creates this file).
# The path below looks for any pre-existing velocity file in SAMPLE_DIR:
_vel_candidates = [f for f in os.listdir(SAMPLE_DIR)
                   if f.endswith("--vel-smooth-with-uncert.csv")] if os.path.isdir(SAMPLE_DIR) else []
VEL_FILE     = os.path.join(SAMPLE_DIR, _vel_candidates[0]) if _vel_candidates \
               else "/path/to/file--vel-smooth-with-uncert.csv"
PARAM_FOLDER = None        # experiment metadata folder (or None)
OUTPUT_DIR   = os.path.join(REPO_ROOT, "examples", "figures", "example_03_output")
MATERIAL     = "Cu"        # must match a key in material_properties section
# ──────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"VEL_FILE  : {VEL_FILE}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print("Ready.")

## 1. Load and visualise the velocity trace

In [ ]:
vdf = pd.read_csv(VEL_FILE, header=0)
t_col, v_col = vdf.columns[0], vdf.columns[1]
u_col = vdf.columns[2] if len(vdf.columns) > 2 else None

t_ns = vdf[t_col].values * 1e9
v    = vdf[v_col].values

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_ns, v, lw=1.5, color='steelblue', label='Velocity')
if u_col:
    u = vdf[u_col].values
    ax.fill_between(t_ns, v - u, v + u, alpha=0.2, color='steelblue', label='±uncertainty')
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Free-surface velocity (m/s)")
ax.set_title("Input velocity trace")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "input_trace.png"), dpi=150)
plt.show()

print(f"Trace duration: {t_ns[-1]-t_ns[0]:.1f} ns,  "
      f"peak velocity ≈ {np.nanmax(v):.1f} m/s")

## 2. Run SPADE (spade_only mode)

In [ ]:
import subprocess
from helix_analysis_toolbox import save_config_to_file

cfg = {
    "cli_settings": {
        "input_files":         None,
        "input_dir":           None,
        "input_pattern":       "*.csv",
        "output_dir":          OUTPUT_DIR,
        "param_folder":        PARAM_FOLDER,
        "analysis_mode":       "spade_only",
        "spade_mode":          "manual",
        "spade_input_files":   [VEL_FILE],
        "spade_input_dir":     None,
        "spade_input_pattern": "*--vel-smooth-with-uncert.csv",
    },
    "alpss_config": {},
    "spade_config": {
        "experiment_velocity_shots": True,
        "experiment_spall_analysis": True,
        "experiment_hel_detection":  True,
        "analysis_model":            "hybrid",
        "spall_detection_method":    "5-segment",
        "spall_start_time_ns":       0.0,
        "spall_end_time_ns":         90.0,
        "threshold_velocity_ms":     5.0,
        "hel_start_time_ns":         0,
        "hel_end_time_ns":           20,
        "minimum_HEL_velocity_expected": 40.0,
        "hel_rdp_epsilon":           1.25,
        "hel_slope_drop_ratio":      0.9,
        "hel_detection_min_points":  10,
        "plot_individual":           True,
        "save_summary":              True,
        "show_plots":                False,
        "skip_unknown_material_traces": False,
    },
    "material_properties": {
        "Cu": {"density": 8960.0, "bulk_wave_speed": 3950.0, "C0": 3950.0, "C_L": 4700.0},
    },
}

cfg_path = os.path.join(OUTPUT_DIR, "spade_only.yml")
save_config_to_file(cfg, cfg_path)

result = subprocess.run(
    [sys.executable, os.path.join(REPO_ROOT, "helix_cli_runner.py"),
     "--config", cfg_path],
    capture_output=False, text=True,
)
print("\nExit code:", result.returncode)

## 3. Read the results

In [ ]:
spade_dir   = os.path.join(OUTPUT_DIR, "SPADE_analysis")
summary_csv = os.path.join(spade_dir, "velocity_shots_summary.csv")

if os.path.exists(summary_csv):
    res = pd.read_csv(summary_csv)
    # Show key columns
    key_cols = [c for c in res.columns if any(k in c.lower() for k in
               ['spall', 'strain', 'shock', 'hel', 'peak', 'material', 'file'])]
    display(res[key_cols].head())
else:
    print("Summary not found — check the run output above.")

## 4. Display the per-trace spall and HEL detection plots

In [ ]:
import glob as _glob
from IPython.display import Image, display as ipy_display

for subfolder, label in [("spall_plots", "Spall detection"),
                          ("HEL_plots",   "HEL detection")]:
    plots = _glob.glob(os.path.join(spade_dir, subfolder, "*.png"))
    if plots:
        print(f"\n── {label} ({len(plots)} plot(s)) ──")
        ipy_display(Image(plots[0], width=800))
    else:
        print(f"No {label.lower()} plots found in {subfolder}/")

## 5. Parameter sensitivity: spall window

Try changing `spall_end_time_ns` and see how it affects the detected
pullback velocity and spall strength.

In [ ]:
# Uncomment and edit to experiment:
# cfg["spade_config"]["spall_end_time_ns"] = 60.0
# save_config_to_file(cfg, cfg_path)
# subprocess.run([sys.executable, os.path.join(REPO_ROOT, "helix_cli_runner.py"),
#                 "--config", cfg_path])
print("Edit the cell above to rerun with different parameters.")

## 6. Spall strength formula reference

$$
\sigma_{\text{spall}} = \frac{1}{2}\, \rho_0\, c_b\, \Delta v_{\text{pullback}}
$$

$$
\Delta \sigma_{\text{spall}} = \frac{1}{2}\, \rho_0\, c_b\,
    \sqrt{\Delta v_{\text{peak}}^2 + \Delta v_{\text{min}}^2}
$$

| Symbol | Meaning |
|--------|---------|
| $\rho_0$ | Initial density (kg/m³) |
| $c_b$ | Bulk wave speed (m/s) |
| $\Delta v_{\text{pullback}}$ | Peak velocity − valley velocity (m/s) |
| $\Delta v_{\text{peak}},\, \Delta v_{\text{min}}$ | Velocity uncertainties at peak and valley |

See `SPALL_STRENGTH_CALCULATION.tex` in `supplementary/references/` for the full derivation.